In [ ]:
import math
import random
import os
import time
import itertools
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils as nn_utils
from torch.utils.data import Dataset, DataLoader
import torch.cuda.amp as amp
from torch.cuda.amp import autocast, GradScaler
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR

In [ ]:
config = {
    'seed': 0,
    'dataset': 'Beauty', # 'Beauty' | 'ML-1M' | 'ML-20M'
    'data_path' : '/gpfs/u/home/TMSR/TMSRvldn/scratch/recsys/Beauty_Ratings.csv',
    'min_user_interactions': 5,
    'max_user_interactions': 200,

    # Training loop
    'num_epochs' : 5,
    'batch_size' : 128,
    'num_workers' : 2,
    'grad_clip' : 1.0,

    # Model hyperparameters
    'max_len' : 20,
    'hidden_units' : 64,
    'num_heads' : 2,
    'num_layers': 2,
    'dropout_rate' : 0.1,
    'mask_prob' : 0.15,

    # Optimizer
    'lr' : 0.001,
    'weight_decay' : 0.001,
}

In [ ]:
seed = config['seed']
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
class MakeSequenceDataSet():

    def __init__(self, config):
        ds = config['dataset']
        # Determine column names and read CSV based on dataset
        if ds == 'Beauty':
            cols = ['UserId', 'ProductId', 'Timestamp']
            self.df = pd.read_csv(config['data_path'])
        elif ds == 'ML-20M':
            cols = ['userId', 'movieId', 'timestamp']
            self.df = pd.read_csv(config['data_path'])
        elif ds == "ML-1M":
            cols = ['userId', 'movieId', 'timestamp']
            # ML-1M uses '::' separator and requires explicit column names
            self.df = pd.read_csv(
                config['data_path'], sep="::", names=cols, engine='python'
            )
        # Filter users by interaction counts if thresholds set
        if (
            config.get('min_user_interactions') is not None
            and config.get('max_user_interactions') is not None
        ):
            user_counts = (
                self.df.groupby(cols[0])
                  .size()
                  .reset_index(name='count')
            )
            valid_users = user_counts[
                (user_counts['count'] >= config['min_user_interactions'])
                & (user_counts['count'] <= config['max_user_interactions'])
            ][cols[0]]
            self.df = self.df[self.df[cols[0]].isin(valid_users)]
        # Build item and user encoders based on unique IDs
        self.item_encoder, self.item_decoder = self.generate_encoder_decoder(cols[1])
        self.user_encoder, self.user_decoder = self.generate_encoder_decoder(cols[0])
        # Store vocabulary sizes
        self.num_item, self.num_user = (
            len(self.item_encoder), len(self.user_encoder)
        )
        # Map original IDs to consecutive indices
        self.df['item_idx'] = self.df[cols[1]].apply(
            lambda x: self.item_encoder[x] + 1
        )
        self.df['user_idx'] = self.df[cols[0]].apply(
            lambda x: self.user_encoder[x]
        )
        # Sort interactions by user and timestamp
        self.df = self.df.sort_values(['user_idx', cols[2]])
        # Split last interaction for validation
        self.user_train, self.user_valid = self.generate_sequence_data()

    def generate_encoder_decoder(self, col: str) -> dict:
        encoder, decoder = {}, {}
        ids = self.df[col].unique()
        for idx, _id in enumerate(ids):
            encoder[_id] = idx
            decoder[idx] = _id
        return encoder, decoder

    def generate_sequence_data(self) -> dict:
        users = defaultdict(list)
        user_train, user_valid = {}, {}
        # Group interactions by user index
        for user_idx, group in self.df.groupby('user_idx'):
            users[user_idx].extend(group['item_idx'].tolist())
        # Reserve last item for validation per user
        for user_idx, seq in users.items():
            user_train[user_idx] = seq[:-1]
            user_valid[user_idx] = [seq[-1]]
        return user_train, user_valid

    def get_train_valid_data(self):
        return self.user_train, self.user_valid

In [ ]:
class BERTRecDataSet(Dataset):

    def __init__(
        self, user_train, max_len, num_user, num_item, mask_prob
    ):
        self.user_train = user_train
        self.max_len = max_len
        self.num_user = num_user
        self.num_item = num_item
        self.mask_prob = mask_prob
        # Precompute set of all item indices for negative sampling
        self._all_items = set(range(1, self.num_item + 1))

    def __len__(self):
        return self.num_user

    def __getitem__(self, user):
        seq = self.user_train[user]
        tokens, labels = [], []
        # Iterate over last max_len items for masking
        for item in seq[-self.max_len:]:
            p = np.random.random()
            if p < self.mask_prob:
                p /= self.mask_prob
                if p < 0.8:
                    # Replace with mask token (last index +1)
                    tokens.append(self.num_item + 1)
                elif p < 0.9:
                    # Replace with random negative sample
                    tokens.extend(
                        self.random_neg_sampling(seq, 1)
                    )
                else:
                    # Keep original item
                    tokens.append(item)
                labels.append(item)
            else:
                tokens.append(item)
                labels.append(0)
        # Pad sequence up to max_len
        pad_len = self.max_len - len(tokens)
        tokens = [0] * pad_len + tokens
        labels = [0] * pad_len + labels
        return torch.LongTensor(tokens), torch.LongTensor(labels)

    def random_neg_sampling(
        self, rated_item: list, num_item_sample: int
    ):  # Negative sample from unseen items
        return random.sample(
            list(self._all_items - set(rated_item)), num_item_sample
        )

In [ ]:
class PositionalEmbedding(nn.Module):

    def __init__(self, max_len, d_model):
        super().__init__()
        self.pe = nn.Embedding(max_len, d_model)

    def forward(self, x):
        batch_size = x.size(0)
        return self.pe.weight.unsqueeze(0).repeat(batch_size, 1, 1)

class TokenEmbedding(nn.Embedding):

    def __init__(self, vocab_size, embed_size=512):
        super().__init__(vocab_size, embed_size, padding_idx=0)

class BERTEmbedding(nn.Module):

    def __init__(
        self, vocab_size, embed_size, max_len, dropout=0.1
    ):
        super().__init__()
        self.token = TokenEmbedding(vocab_size, embed_size)
        self.position = PositionalEmbedding(max_len, embed_size)
        self.dropout = nn.Dropout(p=dropout)
        self.embed_size = embed_size

    def forward(self, sequence):
        # Sum token and position embeddings
        emb = self.token(sequence) + self.position(sequence)
        return self.dropout(emb)

In [ ]:
class Attention(nn.Module):
    def forward(self, query, key, value, mask=None, dropout=None):
        # Compute raw attention scores: QKᵀ / sqrt(d_k)
        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(query.size(-1))
        # Apply causal or padding mask if provided
        if mask is not None:
            # mask shape should broadcast to scores
            scores = scores.masked_fill(mask == 0, float('-inf'))
        # Convert scores to probabilities with softmax
        p_attn = F.softmax(scores, dim=-1)
        # Apply dropout to attention weights
        if dropout is not None:
            p_attn = dropout(p_attn)
        # Weight values by attention probabilities
        return torch.matmul(p_attn, value), p_attn

class MultiHeadedAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1):
        super().__init__()
        assert d_model % h == 0, "d_model must be divisible by num_heads"
        self.d_k = d_model // h
        self.h = h
        # Linear projections for query, key, value
        self.linear_layers = nn.ModuleList([
            nn.Linear(d_model, d_model) for _ in range(3)
        ])
        # Final linear layer after concatenating heads
        self.output_linear = nn.Linear(d_model, d_model)
        self.attention = Attention()
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        # Project and reshape for multi-head attention
        query, key, value = [
            lin(x).view(batch_size, -1, self.h, self.d_k)
                 .transpose(1, 2)
            for lin, x in zip(self.linear_layers, (query, key, value))
        ]
        # Compute scaled dot-product attention
        x, attn = self.attention(query, key, value, mask=mask, dropout=self.dropout)
        # Concatenate heads and project back to model dimension
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.h * self.d_k)
        return self.output_linear(x)

In [ ]:
class GELU(nn.Module):

    def forward(self, x):
        # Gaussian Error Linear Unit activation
        return 0.5 * x * (1 + torch.tanh(math.sqrt(2 / math.pi) *
                                         (x + 0.044715 * x.pow(3))))

class PositionwiseFeedForward(nn.Module):

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        # First linear projects to larger dimension d_ff
        self.w_1 = nn.Linear(d_model, d_ff)
        # Second linear projects back to model dimension
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = GELU()  # Nonlinear activation

    def forward(self, x):
        # Apply FFN: linear→GELU→dropout→linear
        return self.w_2(self.dropout(self.activation(self.w_1(x))))

class LayerNorm(nn.Module):

    def __init__(self, features, eps=1e-6):
        super().__init__()
        # Scale and shift parameters
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        # Normalize across last dimension
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

class SublayerConnection(nn.Module):

    def __init__(self, size, dropout):
        super().__init__()
        # Layer norm + residual dropout wrapper
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        # Apply norm → sublayer → dropout → add residual
        return x + self.dropout(sublayer(self.norm(x)))

In [ ]:
class TransformerBlock(nn.Module):

    def __init__(self, hidden, attn_heads, feed_forward_hidden, dropout):
        super().__init__()
        # Multi-head attention + FFN with residual connections
        self.attention = MultiHeadedAttention(attn_heads, hidden, dropout)
        self.feed_forward = PositionwiseFeedForward(hidden, feed_forward_hidden, dropout)
        self.input_sublayer = SublayerConnection(hidden, dropout)
        self.output_sublayer = SublayerConnection(hidden, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        # Apply attention sublayer
        x = self.input_sublayer(x, lambda _x: self.attention(_x, _x, _x, mask=mask))
        # Apply feed-forward sublayer
        x = self.output_sublayer(x, self.feed_forward)
        return self.dropout(x)

class BERT(nn.Module):

    def __init__(self, bert_max_len, num_items, bert_num_blocks,
                 bert_num_heads, bert_hidden_units, bert_dropout):
        super().__init__()
        vocab_size = num_items + 2        # +1 for mask token, +1 for padding
        hidden = bert_hidden_units
        # Token + positional embeddings
        self.embedding = BERTEmbedding(vocab_size, hidden, bert_max_len, bert_dropout)
        # Stack of transformer blocks
        self.transformer_blocks = nn.ModuleList(
            [TransformerBlock(hidden, bert_num_heads, hidden * 4, bert_dropout)
             for _ in range(bert_num_blocks)]
        )
        # Final linear layer to predict next-item logits
        self.out = nn.Linear(hidden, num_items + 1)

    def forward(self, x):
        # Build mask for padding positions
        mask = (x > 0).int()
        # Embed tokens and positions
        x = self.embedding(x)
        # Pass through transformer layers
        for block in self.transformer_blocks:
            x = block(x, mask)
        # Project to vocabulary size (excluding padding index)
        return self.out(x)

    def init_weights(self):
        # Weight initialization placeholder
        pass

In [ ]:
def train(model, criterion, optimizer, data_loader, scheduler=None, config=None, device='cuda'):
    model.train()  # Enable training mode (dropout, etc.)
    total_loss = 0.0

    for seq, labels in tqdm(data_loader):
        # Move batch to device
        seq, labels = seq.to(device), labels.to(device)

        optimizer.zero_grad()         # Clear previous gradients
        logits = model(seq)           # Forward pass
        # Reshape for cross-entropy: [B, S, V] → [B*S, V]
        logits = logits.view(-1, logits.size(-1))
        labels = labels.view(-1)      # Flatten labels to [B*S]

        loss = criterion(logits, labels)  # Compute masked cross-entropy
        loss.backward()                   # Backpropagate
        # Clip gradients to prevent explosion
        nn_utils.clip_grad_norm_(model.parameters(), config['grad_clip'])
        optimizer.step()                  # Update weights

        if scheduler is not None:
            scheduler.step()              # Adjust learning rate

        total_loss += loss.item()

    # Average loss over batches
    return total_loss / len(data_loader)

def evaluate(model, user_train, user_valid, max_len, data_loader, bert4rec_dataset, make_sequence_dataset):
    model.eval()  # Switch to evaluation mode
    NDCG = 0.0
    HIT = 0.0
    num_item_sample = 100  # Negatives per positive
    users = list(range(make_sequence_dataset.num_user))
    if config['dataset'] == 'ML-20M':
        # Sample subset for faster evaluation
        users = random.sample(users, 5000)

    for user in tqdm(users):
        # Nuild input sequence with mask token at end
        seq = (user_train[user] + [make_sequence_dataset.num_item + 1])[-max_len:]
        seq = [0] * (max_len - len(seq)) + seq  # Left-pad with zeros
        # Candidate items = true next + negatives
        rated = user_train[user] + user_valid[user]
        items = user_valid[user] + bert4rec_dataset.random_neg_sampling(rated, num_item_sample)
        with torch.no_grad():
            seq_tensor = torch.LongTensor([seq]).to(device)
            # Invert logits for ranking (lower is better)
            scores = -model(seq_tensor)[0, -1, items]
            # Compute rank of true item
            rank = scores.argsort().argsort()[0].item()

        if rank < 10:
            NDCG += 1 / np.log2(rank + 2)  # Discounted gain
            HIT += 1

    # Average metrics over users
    NDCG /= len(users)
    HIT /= len(users)
    return NDCG, HIT

In [ ]:
# Instantiate dataset and dataloader
make_sequence_dataset = MakeSequenceDataSet(config)

user_train, user_valid = make_sequence_dataset.get_train_valid_data()

bert4rec_dataset = BERTRecDataSet(
    user_train, config['max_len'],
    make_sequence_dataset.num_user,
    make_sequence_dataset.num_item,
    config['mask_prob']
)

data_loader = DataLoader(
    bert4rec_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    pin_memory=True,
    num_workers=config['num_workers']
)

# Setup device and model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = BERT(
    bert_max_len=config['max_len'],
    num_items=make_sequence_dataset.num_item,
    bert_num_blocks=config['num_layers'],
    bert_num_heads=config['num_heads'],
    bert_hidden_units=config['hidden_units'],
    bert_dropout=config['dropout_rate']
).to(device)

In [ ]:
# Loss, optimizer, and scheduler
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])

def get_scheduler(optimizer, warmup_steps, total_steps):
    # Build a learning rate schedule with linear warmup followed by linear decay
    def lr_lambda(current_step):
        if current_step < warmup_steps:
            # Linearly ramp up LR from 0 to 1 over warmup_steps
            return current_step / max(1, warmup_steps)
        # After warmup, linearly decay LR to 0
        return max(
            0.0,
            1.0 - (current_step - warmup_steps) / max(1, total_steps - warmup_steps)
        )
    # Wrap the lambda function in a PyTorch scheduler
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

total_steps = len(data_loader) * config['num_epochs']
warmup_steps = int(0.1 * total_steps)
scheduler = get_scheduler(optimizer, warmup_steps, total_steps)

In [ ]:
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

# Training loop with metric tracking
start_time = time.time()
loss_list, ndcg_list, hit_list, memory_list = [], [], [], []
for epoch in tqdm(range(1, config['num_epochs'] + 1)):
    torch.cuda.reset_peak_memory_stats(device)
    mem_before = torch.cuda.memory_allocated(device) / 1024**2
    print(f"Epoch {epoch} - Memory before: {mem_before:.2f} MB")

    # Training step
    train_loss = train(model, criterion, optimizer, data_loader, scheduler, config, device)
    loss_list.append(train_loss)
    print(f"Epoch {epoch} | Train loss: {train_loss:.5f}")

    # Record peak GPU memory
    peak_mem = torch.cuda.max_memory_allocated(device) / 1024**2
    memory_list.append(peak_mem)
    print(f"Epoch {epoch} - Peak memory: {peak_mem:.2f} MB")

    # Evaluation step
    ndcg, hit = evaluate(model, user_train, user_valid, config['max_len'], None, bert4rec_dataset, make_sequence_dataset)
    ndcg_list.append(ndcg)
    hit_list.append(hit)
    print(f"NDCG@10: {ndcg:.4f} | HIT@10: {hit:.4f}")

# Final timing and best metrics
total_time = time.time() - start_time
print(f"Total training time: {total_time:.2f} s")
print(f"Best NDCG@10: {max(ndcg_list):.4f}")
print(f"Best HIT@10: {max(hit_list):.4f}")
print(f"Peak GPU Memory: {max(memory_list):.2f} MB")

# Save model checkpoint
torch.save(model.state_dict(), "final_checkpoint.pth")

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(20, 5))
epochs_range = list(range(1, len(loss_list) + 1))

ax[0].plot(epochs_range, loss_list, marker='o')
ax[0].set_title('Loss')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Loss')

ax[1].plot(epochs_range, ndcg_list, marker='o')
ax[1].set_title('NDCG')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('NDCG@10')

ax[2].plot(epochs_range, hit_list, marker='o')
ax[2].set_title('HIT')
ax[2].set_xlabel('Epoch')
ax[2].set_ylabel('HIT@10')

ax[3].plot(epochs_range, memory_list, marker='o')
ax[3].set_title('Peak GPU Memory Usage')
ax[3].set_xlabel('Epoch')
ax[3].set_ylabel('Memory (MB)')

plt.tight_layout()
plt.show()